![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Week 2, Day 3 -- Lab 2: Image Segmentation with a pretrained U-Net

Detection drew a **box** around each object. **Segmentation** is even more
precise: it labels **every pixel** -- exactly which pixels belong to the object
and which are background.

We'll use a **U-Net**, the classic segmentation model. Ours is **already
trained** to highlight abnormal regions in brain MRI scans (transfer learning
again -- we just reuse it).

## 1) Load a pretrained U-Net

In [ ]:
import torch

model = torch.hub.load("mateuszbuda/brain-segmentation-pytorch", "unet",
                       in_channels=3, out_channels=1, init_features=32, pretrained=True)
model.eval();

## 2) Get a brain MRI image

In [ ]:
import urllib.request

url = "https://github.com/mateuszbuda/brain-segmentation-pytorch/raw/master/assets/TCGA_CS_4944.png"
urllib.request.urlretrieve(url, "brain.png")

## 3) Run the model

The U-Net looks at the image and returns a **mask**: for every pixel it decides *object* or *background*.

In [ ]:
import numpy as np
from PIL import Image
from torchvision import transforms

img = Image.open("brain.png")
mean, std = np.mean(img, axis=(0, 1)), np.std(img, axis=(0, 1))
x = transforms.Normalize(mean, std)(transforms.ToTensor()(img)).unsqueeze(0)

with torch.no_grad():
    mask = model(x)[0, 0] > 0.5      # True where the model sees an abnormal region

## 4) See the segmentation

The highlighted pixels are what the U-Net marked.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(img); ax[0].set_title("MRI scan"); ax[0].axis("off")
ax[1].imshow(img); ax[1].imshow(mask, alpha=0.4, cmap="autumn")
ax[1].set_title("U-Net segmentation"); ax[1].axis("off")
plt.show()

## Recap

Three ways a computer can understand an image -- all using a **pretrained**
model (transfer learning):

- **Classification** -- *what* is in the image
- **Detection** (YOLO) -- *what* and *where* (a box)
- **Segmentation** (U-Net) -- *which pixels* (a mask)

### *Contributed By: Sattam Altwaim*